# Norris Phase 4 — devtest S0 baseline and main post-editing matrix (24,288 outputs)

**Start only after** the freeze package was sent to the PI and `config/frozen/protocol_freeze.json` is committed on the `phase4-main` branch.
Runtime: **A100 GPU, High-RAM**. Secrets: `HF_TOKEN`, `OPENAI_API_KEY`.

Everything is resumable: if Colab disconnects, rerun cell 1–2 and then the cell that stopped; completed jobs are reused. **Nothing is scored here.**
GPT note: Tier-1 accounts allow ~10,000 gpt-4o-mini requests/day; cell 7 runs one direction at a time (3,036 requests each). If it stops with HTTP 429 (daily limit), continue the next day.

In [ ]:
# 1 — Mount Drive, clone the Norris repo at REF, load Colab Secrets (never printed).
from pathlib import Path
import os, subprocess, sys, json, hashlib, zipfile
from google.colab import drive, userdata
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/norris')
REPO = Path('/content/csudh-norris-ape-2026')
REF = 'phase4-main'
URL = 'https://github.com/SHAZAN01/csudh-norris-ape-2026.git'
import shutil
shutil.rmtree(REPO, ignore_errors=True)  # always a fresh clone: robust to stale folders and rewritten branches
subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('REPO COMMIT:', COMMIT)
assert (REPO / 'src/common/norris_protocol.py').exists(), 'Wrong branch content: push the patched branch first.'
for name in ['HF_TOKEN', 'OPENAI_API_KEY']:
    try:
        os.environ[name] = userdata.get(name) or ''
    except Exception:
        os.environ[name] = ''
    print(name, 'available' if os.environ[name] else 'MISSING (enable it in Colab Secrets if this notebook needs it)')
LOGS = ROOT / 'outputs/logs/colab'
LOGS.mkdir(parents=True, exist_ok=True)

def sh(cmd, log='run.log'):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd)[:240], flush=True)
    env = os.environ.copy()
    for k in ['PYTHONPATH', 'UV_SYSTEM_PYTHON', 'VIRTUAL_ENV']:
        env.pop(k, None)
    env['TOKENIZERS_PARALLELISM'] = 'false'
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as p, \
            open(LOGS / log, 'a', encoding='utf-8') as f:
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f'Stopped (exit {p.returncode}). Copy the error above and send it; log: {LOGS / log}')

sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
def venv(path, py):
    if not Path(path, 'bin/python').exists():
        sh([sys.executable, '-m', 'uv', 'venv', '--python', py, path])
    return Path(path, 'bin/python')
def uv(py, *pkgs):
    sh([sys.executable, '-m', 'uv', 'pip', 'install', '--python', py, *pkgs], log='install.log')
print('SETUP OK')

In [ ]:
# 2 — Verify the declared freeze and that every frozen source file in this checkout matches it
FREEZE = ROOT / 'outputs/frozen/protocol_freeze.json'
F = json.loads(FREEZE.read_text())
assert F['freeze_declared'] is True, 'Freeze not declared.'
assert FREEZE.read_bytes() == (REPO / 'config/frozen/protocol_freeze.json').read_bytes(), 'Repo and Drive freeze differ.'
bad = [f for f, h in F['code_sha256'].items() if hashlib.sha256((REPO / f).read_bytes()).hexdigest() != h]
assert not bad, f'Frozen code changed: {bad}'
print('FREEZE OK', hashlib.sha256(FREEZE.read_bytes()).hexdigest(), '| declared', F['declared_utc'])
# Preflight: every input must exist before any GPU work (nothing is written here)
_v3 = Path(json.loads((ROOT / 'outputs/phase3_post_editing/rte_v3_latest.json').read_text())['jobs_dir'])
_v2 = Path(json.loads((_v3 / 'prepared_manifest.json').read_text())['v2_prepared_dir'])
_need = [ROOT / 'outputs/logs/data_manifest.json',
         ROOT / 'outputs/generations/baselines/phase3_dev_f4a3af94ba1e72fd/generation_summary.json',
         ROOT / 'outputs/phase4/gate_dev_segmentation.json', ROOT / 'outputs/phase4/gate_dev_generation.json',
         *[_v2 / f'embeddings_{l}.npy' for l in ['eng_Latn', 'tgl_Latn', 'spa_Latn']],
         *[ROOT / f'outputs/generations/baselines/nllb3.3b_{s}_{t}_devtest.jsonl' for s, t in
           [('eng_Latn', 'tgl_Latn'), ('tgl_Latn', 'eng_Latn'), ('eng_Latn', 'spa_Latn'), ('spa_Latn', 'eng_Latn')]]]
_missing = [str(p) for p in _need if not p.exists()]
assert not _missing, f'Missing inputs (send this list): {_missing}'
print('PREFLIGHT OK:', len(_need), 'inputs present')

In [ ]:
# 3 — Environments

# NLLB environment (matches the measured dev baseline run: Python 3.13, torch 2.11.0+cu130, transformers 5.18.0)
NLLB_PY = venv('/content/env_nllb', '3.13')
uv(NLLB_PY, 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu130')
uv(NLLB_PY, 'torch==2.11.0', 'transformers==5.18.0', 'tokenizers==0.23.2', 'sentencepiece==0.2.2',
   'huggingface_hub==1.33.0', 'pysbd==0.3.4')
sh([NLLB_PY, '-c', "import sys, torch, transformers; print(sys.version.split()[0], torch.__version__, transformers.__version__, torch.cuda.is_available())"])

# Retrieval environment (same as the measured pilot retrieval)
RETRIEVAL_PY = venv('/content/env_retrieval', '3.11')
uv(RETRIEVAL_PY, 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
uv(RETRIEVAL_PY, 'torch==2.6.0', 'sentence-transformers==3.4.1', 'transformers==4.49.0', 'huggingface-hub==0.36.0',
   'numpy==1.26.4', 'faiss-cpu==1.9.0', 'pydantic==2.12.3', 'scipy==1.13.1', 'scikit-learn==1.6.1', 'setuptools==80.9.0')
sh([RETRIEVAL_PY, '-c', "import torch, faiss, sentence_transformers; assert torch.cuda.is_available(); print('retrieval env ok')"])

# Llama/GPT environment (vLLM 0.11.2 exactly as in the measured pilot; openai 1.109.1 for AsyncOpenAI)
LLAMA_PY = venv('/content/env_llama', '3.11')
uv(LLAMA_PY, 'vllm==0.11.2', 'transformers==4.57.1', 'pydantic==2.12.3', 'openai==1.109.1')
sh([LLAMA_PY, '-c', "import torch, vllm, pydantic, openai; from vllm.sampling_params import StructuredOutputsParams; "
    "print('vllm', vllm.__version__, 'torch', torch.__version__, 'openai', openai.__version__); assert torch.cuda.is_available()"])

In [ ]:
# 4 — S0: regenerate devtest unsplit control (must equal Phase 2 archive exactly), then the frozen sentence split. ≈8 min.
sh([NLLB_PY, REPO / 'src/baselines/generate_devtest_baseline.py', '--stage', 'devtest', '--root', ROOT, '--freeze', FREEZE],
   log='phase4_s0.log')

In [ ]:
# 5 — Prepare 12,144 frozen jobs (LaBSE/FAISS from the full 997-item dev pool; exact-duplicate exclusion).
sh([RETRIEVAL_PY, REPO / 'src/post_editing/prepare_main_jobs.py', '--root', ROOT, '--freeze', FREEZE], log='phase4_prepare.log')
JOBS = Path(json.loads((ROOT / 'outputs/phase4/main_prepared_latest.json').read_text())['jobs_dir'])
GEN = ROOT / 'outputs/generations/main'
ENGINE = REPO / 'src/post_editing/run_main_experiments.py'
LOGF = ROOT / 'outputs/logs/phase4_execution.log'
print(JOBS)

In [ ]:
# 6 — Smoke stage (2 devtest IDs x 4 directions x 3 strategies = 24 per model). Send the printed SMOKE lines if anything is unusable.
for model in ['gpt4o-mini', 'llama3-8b']:
    sh([LLAMA_PY, ENGINE, '--jobs-dir', JOBS, '--model', model, '--stage', 'smoke', '--generations-dir', GEN, '--log', LOGF],
       log='phase4_smoke.log')

In [ ]:
# 7 — Llama full matrix (12,144 outputs; ≈45–75 min on A100)
sh([LLAMA_PY, ENGINE, '--jobs-dir', JOBS, '--model', 'llama3-8b', '--stage', 'full', '--generations-dir', GEN, '--log', LOGF],
   log='phase4_llama.log')

In [ ]:
# 8 — GPT-4o Mini full matrix, one direction at a time (≈13 min each at 4 req/s; total ≈ $1.5–2)
for direction in ['eng-tag', 'tag-eng', 'eng-spa', 'spa-eng']:
    sh([LLAMA_PY, ENGINE, '--jobs-dir', JOBS, '--model', 'gpt4o-mini', '--direction', direction, '--stage', 'full',
        '--generations-dir', GEN, '--log', LOGF, '--max-estimated-usd', '5'], log='phase4_gpt.log')

In [ ]:
# 9 — Completion audit (24 conditions x 1,012 unique IDs, provenance, no unresolved API states)
sh([LLAMA_PY, REPO / 'src/post_editing/phase4_summary.py', '--root', ROOT, '--freeze', FREEZE], log='phase4_summary.log')
print(json.dumps({k: v for k, v in json.loads((ROOT / 'outputs/phase4/phase4_summary.json').read_text()).items()
                  if k in ['status', 'checks']}, indent=1))

In [ ]:
# 10 — Evidence ZIP for Report 4 (numeric summaries + specs + logs; raw generations stay on Drive)
from google.colab import files
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zp = ROOT / f'outputs/reports/Norris_Phase4_Evidence_{stamp}.zip'
s0 = Path(json.loads((ROOT / 'outputs/phase4/devtest_baseline_latest.json').read_text())['run_dir'])
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [ROOT / 'outputs/phase4/phase4_summary.json', ROOT / 'outputs/phase4/phase4_conditions.csv',
              JOBS / 'prepared_manifest.json', s0 / 'run_spec.json', s0 / 'generation_summary.json', FREEZE, LOGF,
              *JOBS.glob('runs/*/*/run_spec.json'), *JOBS.glob('runs/*/*/run_summary.json'),
              *JOBS.glob('runs/*/*/smoke_summary.json'), *JOBS.glob('runs/*/*/requirements_resolved.txt'), *LOGS.glob('phase4*.log')]:
        if p.exists():
            z.write(p, str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else p.name)  # unique paths
print(zp); files.download(str(zp))